# 04 - Advanced Search Strategies

If you want to estimate a large number of models, it is recommended to run Delphos in a loop. Delphos caches Apollo estimation results in a local SQLite database, meaning that if it proposes the same model twice, it will not waste time estimating it again in R.

In this notebook, we'll see how to:
- Configure advanced parameters.
- Run Delphos in a loop.
- Save results to CSV incrementally.


In [1]:
import delphos as dp
import pandas as pd

dataset = dp.load_dataset("dataset_4")
agent = dp.load_agent()


## 1. Advanced Parameters

Delphos exposes advanced sampling controls via `**advanced_kwargs` in the `propose()` method. 

- `epsilon`: (default `0.0`) Controls exploration vs exploitation. E.g. `epsilon=0.1` means 10% of the time, the agent takes a random action.
- `temperature`: (default `1.0`) Softmax temperature for sampling. Higher = more uniform, lower = more greedy.


In [2]:
models = agent.propose(
    dataset,
    n_models=2,
    estimate=True,
    epsilon=0.05,       # 5% random exploration
    temperature=1.5     # slightly softer sampling
)
models.to_dataframe()


,task_id,task_name,specification_key,episode_length,search_strategy,attempt_found,estimated,reward,n_terms,action_indices,...,rho2_0,adjRho2_0,rho2_C,adjRho2_C,AIC,BIC,eigValue,timeTaken,nFreeParams,skipped
0,4,Swissmetro,1110_2212_3212_4222_5000_6126_7000,10,topk,0,True,0.272179,5,"[125, 21, 106, 73, 17, 75, 149, 27, 131, 215]",...,0.272206,0.268601,0.141389,0.137562,8116.013783,8247.930173,-0.895512,1.737639,20,0
1,4,Swissmetro,1110_2120_3212_4111_5000_6110_7000,10,topk,1,True,0.251856,5,"[215, 17, 125, 201, 21, 106, 27, 74, 75, 17]",...,0.250932,0.248769,0.116292,0.114165,8336.081331,8415.231165,-2.507729,1.044467,12,0


## 2. Strategy Schedules

Instead of using the same search parameters over and over, you can define a schedule to vary them. A common pattern is to start greedy, move to top-k sampling for diverse candidates, and finish with stochastic sampling to force exploration.

In [3]:
strategy_schedule = [
    {"strategy": "greedy", "n_models": 1, "max_attempts": 10},
    {"strategy": "topk", "n_models": 2, "max_attempts": 100, "top_k": 5, "temperature": 0.8},
    {"strategy": "stochastic", "n_models": 2, "max_attempts": 100, "epsilon": 0.15},
]
strategy_schedule

[{'strategy': 'greedy', 'n_models': 1, 'max_attempts': 10},
 {'strategy': 'topk',
  'n_models': 2,
  'max_attempts': 100,
  'top_k': 5,
  'temperature': 0.8},
 {'strategy': 'stochastic',
  'n_models': 2,
  'max_attempts': 100,
  'epsilon': 0.15}]

## 3. Running a batch loop

To search extensively, place the proposal call inside a loop and append the results to a CSV. We rotate through the strategy schedule at each iteration.

In [4]:
import os

output_csv = "advanced_search_results.csv"

n_iterations = 3
batch_size = 2

for i in range(n_iterations):
    settings = strategy_schedule[i % len(strategy_schedule)]
    print(f"Iteration {i+1}/{n_iterations} using settings: {settings}")
    
    batch = agent.propose(
        dataset,
        estimate=True,
        **settings
    )
    
    df = batch.to_dataframe()
    
    if not os.path.exists(output_csv):
        df.to_csv(output_csv, index=False)
    else:
        df.to_csv(output_csv, mode='a', header=False, index=False)

print("Search completed!")

Iteration 1/3 using settings: {'strategy': 'greedy', 'n_models': 1, 'max_attempts': 10}
Iteration 2/3 using settings: {'strategy': 'topk', 'n_models': 2, 'max_attempts': 100, 'top_k': 5, 'temperature': 0.8}


Iteration 3/3 using settings: {'strategy': 'stochastic', 'n_models': 2, 'max_attempts': 100, 'epsilon': 0.15}


2026-09-16 12:46:43,002 [ERROR] Delphos.apollo: Estimation failed: 1110_2124_3322_4213_5000_6110_7000
Traceback (most recent call last):
  File "/Users/gnova/Developer/Main-Delphos/Delphos/src/delphos/env/apollo/estimator.py", line 122, in run_apollo_estimation
    subprocess.run(
  File "/Users/gnova/.pyenv/versions/3.12.11/lib/python3.12/subprocess.py", line 571, in run
    raise CalledProcessError(retcode, process.args,
subprocess.CalledProcessError: Command '['Rscript', '/Users/gnova/Developer/Main-Delphos/Delphos/src/delphos/data/bundled/datasets/dataset_4/outputs/1110_2124_3322_4213_5000_6110_7000_estimation.R']' returned non-zero exit status 1.


EXCEPTION CAUGHT IN evaluate_specification: Rscript failed during estimation:
Warning message:
package ‘apollo’ was built under R version 4.5.2 
Error in if (any(test_gradient == 0)) { : 
  missing value where TRUE/FALSE needed
Calls: apollo_estimate
Execution halted

Search completed!


Traceback (most recent call last):
  File "/Users/gnova/Developer/Main-Delphos/Delphos/src/delphos/env/apollo/estimator.py", line 122, in run_apollo_estimation
    subprocess.run(
  File "/Users/gnova/.pyenv/versions/3.12.11/lib/python3.12/subprocess.py", line 571, in run
    raise CalledProcessError(retcode, process.args,
subprocess.CalledProcessError: Command '['Rscript', '/Users/gnova/Developer/Main-Delphos/Delphos/src/delphos/data/bundled/datasets/dataset_4/outputs/1110_2124_3322_4213_5000_6110_7000_estimation.R']' returned non-zero exit status 1.

The above exception was the direct cause of the following exception:

Traceback (most recent call last):
  File "/Users/gnova/Developer/Main-Delphos/Delphos/src/delphos/env/environment.py", line 92, in evaluate_specification
    outcome = run_apollo_estimation(
              ^^^^^^^^^^^^^^^^^^^^^^
  File "/Users/gnova/Developer/Main-Delphos/Delphos/src/delphos/env/apollo/estimator.py", line 133, in run_apollo_estimation
    raise Runtime

## 4. The Result Cache

Why is it safe to loop like this overnight? Because Delphos maintains an internal SQLite database caching all estimation results! If it proposes a specification it has already seen, it immediately loads the results instead of calling R, saving an enormous amount of time.

In [5]:
from delphos.env.result_cache import ResultCache

cache = ResultCache(dataset.rewards_path)
print("Cache path:", cache.db_path)
print("Rows currently stored:", len(cache.load(dataset.name)))


Cache path: /Users/gnova/Developer/Main-Delphos/Delphos/src/delphos/data/bundled/datasets/dataset_4/rewards.sqlite
Rows currently stored: 48


## 5. Reviewing unique results

Once your search finishes, load the CSV and drop duplicates to find the best models!

In [6]:
results = pd.read_csv(output_csv)

unique_results = results.drop_duplicates(subset=["specification_key"])
best_models = unique_results.sort_values("BIC", ascending=True)

print(f"Found {len(unique_results)} unique specifications.")
best_models[["specification_key", "BIC", "AIC", "LLout"]].head(5)


Found 5 unique specifications.


,specification_key,BIC,AIC,LLout
2,1110_2212_3210_4111_5000_6110_7000,8188.248742,8122.290547,-4051.145274
1,1110_2124_3212_4111_5000_6110_7000,8257.960248,8159.022955,-4064.511478
3,1110_2212_3110_4124_5000_6110_7000,8339.406605,8260.256771,-4118.128385
0,1110_2212_3210_4214_5000_6110_7000,10291.740385,10223.540776,-5101.770388
4,1110_2124_3322_4213_5000_6110_7000,NaN,NaN,NaN
